# 03 | Constrained Portfolio Construction

**Research setting:** retrospective replay of the third-tranche screening decision.  
**Historical decision date:** 3 November 2026; **market-data cutoff:** close of 2 November 2026.  
**Purpose:** compare constrained combinations of six candidates with an equal-exposure benchmark. This is a **research replay**, not an order recommendation.

The uploaded working notebook screened 14 candidates and evaluated \(inom{14}{6}=3003\) combinations. It also imposed internal rules: one new energy producer among ENI/Aker BP/Equinor; at least five distinct sectors in the new tranche; at most 5% of NAV per name; at most 15% of NAV per sector. None of these limits should be labeled as an official Euronext point formula.

**Data requirement:** `prices_eur_2026-11-04.csv` or another consistent EUR price panel. To avoid look-ahead bias, this notebook **truncates all data at 2026-11-02**. Scores are recomputed from information available at that cutoff; consequently results can differ from previous screens if the original saved ranking applied a different convention.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run from the repository root, or from within notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
OUTPUT = ROOT / "reports" / "research_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT.resolve())
print("Data directory:", DATA.resolve())

In [ ]:
PRICE_FILE = DATA / "prices_eur_2026-11-04.csv"
if not PRICE_FILE.is_file():
    raise FileNotFoundError(
        f"Missing {PRICE_FILE}. Copy your previously validated EUR-adjusted "
        "price panel into data/ (or update PRICE_FILE). Do not commit private "
        "or third-party price datasets without reviewing distribution rights."
    )
prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
prices.index = pd.to_datetime(prices.index).normalize()
prices = prices.apply(pd.to_numeric, errors="coerce").sort_index()
if prices.index.has_duplicates or prices.columns.has_duplicates:
    raise ValueError("Duplicate timestamps or ticker identifiers")
if prices.isna().any().any() or (prices <= 0).any().any():
    raise ValueError("Missing or nonpositive values in the EUR price panel")
if prices.shape[0] < 254:
    raise ValueError("At least 254 prices required for a 252-return risk window")
returns = prices.pct_change(fill_method=None).dropna()
print(f"{prices.shape[1]} securities, {prices.shape[0]} dates; last = {prices.index[-1].date()}")

In [ ]:
CUTOFF = pd.Timestamp("2026-11-02")
prices = prices.loc[:CUTOFF]
if prices.index[-1] != CUTOFF:
    raise ValueError("Historical cutoff 2026-11-02 absent in data")
returns = prices.pct_change(fill_method=None).dropna().tail(252)
NAV, EQUITY, CASH = 50340.30, 20394.84, 29945.46
TARGET = 0.65
assert abs(NAV - EQUITY - CASH)<0.01
old_quantities = pd.Series({'AI.PA': 8, 'ENX.PA': 1, 'HEIA.AS': 29, 'TTE.PA': 26, 'G.MI': 52, 'SRG.MI': 336, 'KPN.AS': 462, 'SAN.PA': 25, 'PRY.MI': 18, 'ML.PA': 62, 'CAP.PA': 15}, dtype=float)
old_sectors = {'AI.PA': 'Materials', 'ENX.PA': 'Financials', 'HEIA.AS': 'Consumer Staples', 'TTE.PA': 'Energy', 'G.MI': 'Financials', 'SRG.MI': 'Utilities', 'KPN.AS': 'Telecommunications', 'SAN.PA': 'Healthcare', 'PRY.MI': 'Industrials', 'ML.PA': 'Consumer Discretionary', 'CAP.PA': 'Technology'}
candidate_sectors = {'OR.PA': 'Consumer Staples', 'CS.PA': 'Financials', 'ENI.MI': 'Energy', 'EDP.LS': 'Utilities', 'RACE.MI': 'Consumer Discretionary', 'AKRBP.OL': 'Energy', 'EQNR.OL': 'Energy', 'TRN.MI': 'Utilities', 'INGA.AS': 'Financials', 'UCB.BR': 'Healthcare', 'CPR.MI': 'Consumer Staples', 'BN.PA': 'Consumer Staples', 'DSY.PA': 'Technology', 'ENEL.MI': 'Utilities'}
if not set(old_quantities.index).issubset(prices.columns):
    raise ValueError("Existing holdings missing from price dataset")
if not set(candidate_sectors).issubset(prices.columns):
    raise ValueError("Candidate tickers missing from price dataset")
marked_old = old_quantities * prices.iloc[-1][old_quantities.index]
old_values = marked_old / marked_old.sum() * EQUITY
BUDGET = 0.65*NAV - EQUITY
EQUAL_TICKET = BUDGET / 6
print("Budget:",round(BUDGET,2)," | Equal candidate ticket:",round(EQUAL_TICKET,2))

## 1. Recompute the 14-candidate factor scores using the same historical cutoff

In the original working notebook, part of the analysis used saved ranking files. Here the data cutoff is explicit, preventing post-2-November information from entering the 3-November decision replay.

In [ ]:
m63 = prices.iloc[-1]/prices.iloc[-64]-1
m126 = prices.iloc[-1]/prices.iloc[-127]-1
vol = returns.std()*np.sqrt(252)
down = np.sqrt(returns.clip(upper=0).pow(2).mean()*252)
score = 100*(0.55*(0.4*m63.rank(pct=True)+0.6*m126.rank(pct=True))
             +0.27*(-down).rank(pct=True)+0.18*(-vol).rank(pct=True))
pool = list(candidate_sectors)
assert len(pool)==14
print(score[pool].sort_values(ascending=False).round(2).to_string())

## 2. Evaluate constant-weight synthetic NAV returns

Daily returns for a hypothetical portfolio are \(r_{P,t}=\sum_i w_{i,NAV}\,r_{i,t}\). Assume the cash return is zero and weights are constant across the historical window. We do **not** rebalance actual trading positions on each historical date. CVaR and drawdown are historical diagnostics based on this hypothetical path.

In [ ]:
from itertools import combinations

def statistics(series):
    daily = np.asarray(series, dtype=float)
    annual_vol = np.std(daily,ddof=1)*np.sqrt(252)
    q = np.quantile(daily,0.05)
    cvar = -np.mean(daily[daily<=q])
    path = np.r_[1.0,np.cumprod(1.0+daily)]
    mdd = np.min(path/np.maximum.accumulate(path)-1)
    return float(annual_vol),float(cvar),float(mdd)

stock_vol = returns.std()*np.sqrt(252)
old_series = returns[old_values.index] @ (old_values / NAV)
benchmark_exposure = TARGET
benchmark_daily = returns[old_values.index] @ (old_values/old_values.sum()) * benchmark_exposure
benchmark_vol,benchmark_cvar,benchmark_mdd=statistics(benchmark_daily)
old_sector_amounts = pd.DataFrame({"Sector":[old_sectors[t] for t in old_values.index],"EUR":old_values.values}).groupby("Sector")["EUR"].sum().to_dict()
energy_group={"ENI.MI","AKRBP.OL","EQNR.OL"}

In [ ]:
records=[]
for combo in combinations(pool,6):
    sec=[candidate_sectors[t] for t in combo]
    if len(set(combo)&energy_group)>1 or len(set(sec))<5:
        continue
    sector_sum=old_sector_amounts.copy()
    for s in sec:
        sector_sum[s]=sector_sum.get(s,0)+EQUAL_TICKET
    if max(sector_sum.values())/NAV > 0.15:
        continue
    if max(float(old_values.max()),EQUAL_TICKET)/NAV >0.05:
        continue
    daily = old_series + returns[list(combo)].sum(axis=1)*EQUAL_TICKET/NAV
    nav_vol,cvar,mdd=statistics(daily)
    if nav_vol > 1.05*benchmark_vol or cvar > 1.05*benchmark_cvar:
        continue
    all_vals = pd.concat([old_values,pd.Series(EQUAL_TICKET,index=combo)])
    w_nav=all_vals/NAV
    dr=float((w_nav @ stock_vol[all_vals.index])/nav_vol)
    eqvol=nav_vol/(all_vals.sum()/NAV)
    records.append({"Combination":" + ".join(combo),"Mean_factor_score":float(score[list(combo)].mean()),
                    "DR":dr,"Vol_NAV_pct":100*nav_vol,"Vol_Equity_pct":100*eqvol,
                    "CVaR95_daily_pct":100*cvar,"MDD_pct":100*mdd,
                    "Sector_count_new":len(set(sec))})
scenarios=pd.DataFrame(records)
if scenarios.empty:
    raise RuntimeError("No candidate portfolios passed the historical constraints")
print("All combinations:",3003,"| Eligible in this replay:",len(scenarios))
print(scenarios.sort_values("DR",ascending=False).head(5).round(3).to_string(index=False))

## 3. Multi-objective ranking, not a predictive return estimate

The same 35/25/25/15 blend used by the historical decision script is reproduced below. It ranks eligible portfolios relative to each other. **Selection on this sample induces data-snooping risk**; no forecast accuracy is established by a high score.

In [ ]:
for col,ascending in [("Mean_factor_score",True),("DR",True),("CVaR95_daily_pct",False),("MDD_pct",True)]:
    scenarios["Rank_"+col]=scenarios[col].rank(pct=True,ascending=ascending)
scenarios["Balanced_Score"] = 100*(0.35*scenarios["Rank_Mean_factor_score"]+
                                 0.25*scenarios["Rank_DR"]+
                                 0.25*scenarios["Rank_CVaR95_daily_pct"]+
                                 0.15*scenarios["Rank_MDD_pct"])
print(scenarios.sort_values("Balanced_Score",ascending=False)[[
    "Combination","Balanced_Score","Mean_factor_score","DR","Vol_Equity_pct","CVaR95_daily_pct","MDD_pct"]].head(8).round(3).to_string(index=False))
scenarios.to_csv(OUTPUT/"03_constrained_scenarios_replay.csv",index=False)
print("Saved reproducible scenario replay in:",OUTPUT)

## Research interpretation and boundaries

- Historical reference results were 3,003 tested; 1,164 structurally eligible; 971 passed the original risk filter. **Differences from those numbers are possible** due to the fresh, explicitly cutoff-based factor calculation and code reimplementation.
- This notebook is a historical *replay*. It cannot establish a causal or out-of-sample improvement.
- Equal ticket sizes are an analytical simplification. Integer share quantities, live execution prices and FX settlement require a separate pre-trade process.
- The research model is different from the older `src/alpha/optimizer.py` mean-variance prototype. Both must be labeled accurately in documentation.